## ۱. داده‌های ساختگی و قرارداد آزمایش
ستون `synthetic_group` در آموزش هیچ مدلی استفاده نمی‌شود؛ فقط برای ممیزی مستقل است.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from chapter08_examples import produce_datasets, manual_example, manual_1d_dbscan, k_medoids_exhaustive, feature_matrix, experiment, assess
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score
frame = pd.read_csv('clustering_synthetic.csv')
print(frame.groupby('dataset').size().to_dict())
print(frame.columns.tolist())

{'compact': 204, 'moons': 214, 'uniform': 155}
['observation_id', 'dataset', 'x1', 'x2', 'synthetic_group']


## ۲. محاسبهٔ دستی شش نقطه
دو مرکز جدید و SSE را با فصل مقایسه کنید.

In [2]:
m = manual_example()
print('centers:', m['new_centers'])
print('SSE:', m['sse'])
assert np.isclose(m['sse'], 8/3)

centers: [[1.33333333 1.33333333]
 [8.33333333 8.33333333]]
SSE: 2.666666666666667


## ۳. اصل انتساب به نزدیک‌ترین مرکز
روی همان شش نقطه، `k-means` را با مراکز مشخص اجرا کنید.

In [3]:
manual_points=m['points']
km6=KMeans(n_clusters=2,init=m['initial_centers'],n_init=1,random_state=824).fit(manual_points)
print('inertia:',km6.inertia_)
print('predicted clusters:',km6.labels_.tolist())

inertia: 2.666666666666667
predicted clusters: [0, 0, 0, 1, 1, 1]


## ۴. فاصلهٔ medoid
این تابع صرفاً جست‌وجوی کامل روی دادهٔ کوچک است و برای مجموعهٔ بزرگ طراحی نشده است.

In [4]:
print(k_medoids_exhaustive([[0],[1],[5]], k=1))

{'cost': 5.0, 'medoid_indices': [1], 'points': [[1.0]]}


## ۵. k-means روی دادهٔ فشرده
ارزیابی داخلی کیفیت به‌معنای برچسب واقعی نیست.

In [5]:
compact,xc=feature_matrix(frame,'compact')
kmeans=KMeans(n_clusters=3,random_state=824,n_init=20).fit(xc)
print('inertia',round(kmeans.inertia_,4))
print('silhouette',round(silhouette_score(xc,kmeans.labels_),4))

inertia 190.6923
silhouette 0.7713


## ۶. حساسیت به تعداد خوشه‌ها
SSE با افزایش k پایین می‌آید ولی پایین‌بودن آن به‌تنهایی معیار انتخاب نیست.

In [6]:
for k in range(1,7):
    model=KMeans(n_clusters=k,random_state=824,n_init=20).fit(xc)
    print(k,round(model.inertia_,2))

1 2173.06
2 1005.56
3 190.69
4 159.18
5 134.77
6 119.76


## ۷. محاسبهٔ دستی DBSCAN
در این قرارداد خود نقطه در تعداد همسایه‌ها محسوب می‌شود.

In [7]:
dbsmall=manual_1d_dbscan(eps=1.1,min_samples=3)
print('core',dbsmall['core'],'border',dbsmall['border'],'noise',dbsmall['noise'])
assert dbsmall['core']==[1]

core [1] border [0, 2] noise [3, 4]


## ۸. هلال‌ها: روش مرکز-محور در برابر چگالی‌محور
خروجی `-1` همان شناسهٔ نویز است، نه گروه منفی.

In [8]:
moons,xm=feature_matrix(frame,'moons')
l_km=KMeans(n_clusters=2,n_init=20,random_state=824).fit_predict(xm)
l_db=DBSCAN(eps=.38,min_samples=5).fit_predict(xm)
print('DBSCAN clusters:',len(set(l_db))-(int(-1 in l_db)))
print('DBSCAN noise:',np.sum(l_db==-1))
print('DBSCAN silhouette, noise excluded:',assess(xm,l_db))

DBSCAN clusters: 2
DBSCAN noise: 12
DBSCAN silhouette, noise excluded: {'silhouette': 0.32381215777218, 'davies_bouldin': 1.173980386638206, 'used': 202, 'excluded_noise': 12}


## ۹. دادهٔ یکنواخت و دام تفسیر
در نبود گروه‌های تعبیه‌شده هم افراز اجباری به دست می‌آید.

In [9]:
uniform,xu=feature_matrix(frame,'uniform')
lab=KMeans(n_clusters=3,random_state=824,n_init=20).fit_predict(xu)
print('silhouette despite no planted groups:',round(silhouette_score(xu,lab),4))

silhouette despite no planted groups: 0.4603


## ۱۰. ارزیابی بیرونی فقط پس از خوشه‌بندی
اگر برای ممیزی برچسب آزمایشگاهی داریم، هرگز نباید آن را به ویژگی‌های ورودی اضافه کنیم.

In [10]:
clean=compact.synthetic_group != 'injected_noise'
print('ARI (audit only):',round(adjusted_rand_score(compact.loc[clean,'synthetic_group'],kmeans.labels_[clean.to_numpy()]),4))

ARI (audit only): 1.0


## ۱۱. پایداری در آغازهای مختلف
ARI با شناسهٔ گروه جابه‌جاشده نیز قابل مقایسه است.

In [11]:
for seed in range(5):
    pred=KMeans(n_clusters=3,random_state=seed,n_init=1).fit_predict(xc)
    print(seed,round(adjusted_rand_score(kmeans.labels_,pred),3))

0 1.0
1 1.0
2 1.0
3 1.0
4 0.985


## ۱۲. ممیزی یکپارچه
همهٔ خروجی‌ها فقط مربوط به همین دادهٔ ساختگی‌اند.

In [12]:
r=experiment(frame)
print('manual_sse:',r['manual_sse'])
print('compact silhouette:',round(r['compact_kmeans_silhouette'],4))
print('moons DBSCAN clusters/noise:',r['moons_dbscan_clusters'],r['moons_dbscan_noise'])
print('uniform null silhouette:',round(r['uniform_kmeans_silhouette'],4))

manual_sse: 2.666666666666667
compact silhouette: 0.7713
moons DBSCAN clusters/noise: 2 12
uniform null silhouette: 0.4603
